In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/models/qwen-lm/qwen-3/transformers/4b/1/model.safetensors.index.json
/kaggle/input/models/qwen-lm/qwen-3/transformers/4b/1/model-00003-of-00003.safetensors
/kaggle/input/models/qwen-lm/qwen-3/transformers/4b/1/config.json
/kaggle/input/models/qwen-lm/qwen-3/transformers/4b/1/merges.txt
/kaggle/input/models/qwen-lm/qwen-3/transformers/4b/1/README.md
/kaggle/input/models/qwen-lm/qwen-3/transformers/4b/1/tokenizer.json
/kaggle/input/models/qwen-lm/qwen-3/transformers/4b/1/vocab.json
/kaggle/input/models/qwen-lm/qwen-3/transformers/4b/1/model-00001-of-00003.safetensors
/kaggle/input/models/qwen-lm/qwen-3/transformers/4b/1/tokenizer_config.json
/kaggle/input/models/qwen-lm/qwen-3/transformers/4b/1/model-00002-of-00003.safetensors
/kaggle/input/models/qwen-lm/qwen-3/transformers/4b/1/generation_config.json
/kaggle/input/competitions/cse-151-b-spring-2026-competition/sample_submission.csv
/kaggle/input/competitions/cse-151-b-spring-2026-competition/private.jsonl
/kaggle/input/co

In [2]:
for root, dirs, files in os.walk("/kaggle/input"):
    if "judger.py" in files:
        print(root)

/kaggle/input/datasets/michi355/judger-and-utils


In [3]:
import json
import os
import torch
import re
import sys

from pathlib import Path
from tqdm import tqdm
from collections import Counter
from transformers import AutoTokenizer, AutoModelForCausalLM

In [4]:
MODEL_ID = "/kaggle/input/models/qwen-lm/qwen-3/transformers/4b/1"
DATA_PATH = "/kaggle/input/competitions/cse-151-b-spring-2026-competition/private.jsonl"
OUTPUT_PATH = "/kaggle/working/submission.jsonl"

BATCH_SIZE = 8  # Reduced for memory stability with 2048 tokens
MAX_TOKENS = 768  # Necessary budget for Thinking models
SAVE_EVAL = False

os.makedirs("results", exist_ok=True)

In [5]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    if "config.json" in files:
        print(root)

/kaggle/input/models/qwen-lm/qwen-3/transformers/4b/1


In [6]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    if "public.jsonl" in files:
        print(os.path.join(root, "public.jsonl"))

/kaggle/input/competitions/cse-151-b-spring-2026-competition/public.jsonl


In [7]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA devices:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
CUDA devices: 2
GPU: Tesla T4


In [8]:
!nvidia-smi

Wed Jun  3 17:50:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
# -- 2. Model & Tokenizer Loading --
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    device_map="auto" 
)
print(f"Model successfully loaded on devices: {model.hf_device_map}")

Loading tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading model...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model successfully loaded on devices: {'model.embed_tokens': 0, 'lm_head': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 0, 'model.layers.15': 0, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.layers.32': 1, 'model.layers.33': 1, 'model.layers.34': 1, 'model.layers.35': 1, 'model.norm': 1, 'model.rotary_emb': 1}


In [10]:
# -- 3. Prompt Engineering --
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. "
    "1. If a question asks for a single number, ONLY output that number. Do not include equations like x=24. "
    "2. If a question asks for an interval, format it exactly as [a, b] or (a, b). "
    "3. If a question has multiple parts, separate the answers with commas inside a SINGLE box: \\boxed{ans1, ans2}. "
    "4. Stop generating immediately after providing the box."
    "Your very last line MUST be exactly: Final Answer: \\boxed{result}"
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert solver. You must be extremely concise. "
    "Output ONLY the letter of the correct option (A, B, C, or D). "
    "DO NOT output reasoning, DO NOT output conversational text. "
    "Format: Final Answer: \\boxed{LETTER}"
)

def clean_latex(text):
    """Removes common LaTeX formatting that breaks string comparisons."""
    text = re.sub(r"\\text\{(.+?)\}", r"\1", text)
    text = text.replace("$", "").replace("\\ ", " ").strip()
    return text

def build_prompt(item):
    question = item["question"]
    options = item.get("options")

    math_examples = (
        "Example 1:\n"
        "Question: What is the derivative of x^2 + 3x at x=2?\n"
        "Answer: The derivative is 2x + 3. Evaluated at x=2, we get 2(2) + 3 = 7.\n"
        "Final Answer: \\boxed{7}\n\n"
        "Example 2:\n"
        "Question: Solve for x: 2x - 4 = 10\n"
        "Answer: Adding 4 to both sides gives 2x = 14. Dividing by 2 gives x = 7.\n"
        "Final Answer: \\boxed{7}\n\n"
    )

    if options:
        # Re-added the logic to build the A, B, C option strings!
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(
            f"{lbl}. {opt.strip()}"
            for lbl, opt in zip(labels, options)
        )
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"

    return SYSTEM_PROMPT_MATH, f"{math_examples}Now solve this:\nQuestion: {question}"
import sympy

def extract_answer_content(text, is_mcq=False):
    boxes = re.findall(r"\\boxed\{(.+?)\}", text)
    if not boxes:
        return text.strip()[-20:]
    
    res = boxes[-1].strip().replace("$", "")
    
    if is_mcq:
        m = re.search(r"([A-Z])", res.upper())
        return m.group(1) if m else res[:1].upper()
    
    try:
        parts = res.split(',')
        evaluated_parts = []
        for p in parts:
            # Clean and evaluate each segment
            clean_p = p.replace("\\dfrac", "").replace("\\frac", "").replace("{", "(").replace("}", ")")
            evaluated_parts.append(str(float(sympy.sympify(clean_p).evalf())))
        
        return evaluated_parts if len(evaluated_parts) > 1 else evaluated_parts[0]
    except:
        return res

In [11]:
def run_fast_inference():
    with open(DATA_PATH, "r") as f:
        data = [json.loads(line) for line in f] 

    final_responses = []

    for i in tqdm(range(0, len(data), BATCH_SIZE), desc="Generating"):
        batch_items = data[i : i + BATCH_SIZE]
        batch_prompts = []

        for item in batch_items:
            sys_p, user_content = build_prompt(item)
            prompt = tokenizer.apply_chat_template(
                [
                    {"role": "system", "content": sys_p},
                    {"role": "user", "content": user_content}, 
                ],
                tokenize=False,
                add_generation_prompt=True,
            )
            batch_prompts.append(prompt)

        inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_TOKENS, 
                do_sample=True,
                temperature=0.1,           
                top_p=0.90,
                use_cache=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        # Decode each generated response in the batch cleanly
        input_len = inputs.input_ids.shape[-1]
        for out in outputs:
            decoded = tokenizer.decode(out[input_len:], skip_special_tokens=True)
            final_responses.append(decoded)

    return data, final_responses

In [12]:
!pip install antlr4-python3-runtime==4.11.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 6.7 MB/s eta 0:00:00
  Attempting uninstall: antlr4-python3-runtime
    Found existing installation: antlr4-python3-runtime 4.9.3
    Uninstalling antlr4-python3-runtime-4.9.3:
      Successfully uninstalled antlr4-python3-runtime-4.9.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.1 which is incompatible.


In [13]:
import sys
sys.path.insert(0, "/kaggle/input/datasets/michi355/judger-and-utils")

from judger import Judger
judger = Judger(strict_extract=False)

print("Judger works!")

Judger works!


In [14]:
# -- 5. Scoring & Extraction --
import pandas as pd

def extract_letter(text):
    m = re.search(r"\\boxed\{([A-Z])\}", text, re.IGNORECASE)
    if m:
        return m.group(1).upper()

    m = re.search(r"(?:Answer|Choice|Option):\s*([A-Z])", text, re.IGNORECASE)
    if m:
        return m.group(1).upper()

    matches = re.findall(r"\b([A-Z])\b", text)
    return matches[-1].upper() if matches else ""


if __name__ == "__main__":
    test_data, model_responses = run_fast_inference()

    sys.path.insert(0, "/kaggle/input/datasets/michi355/judger-and-utils")

    try:
        from judger import Judger
        judger = Judger(strict_extract=False)
    except ImportError:
        judger = None
        print("Judger module not found.")

    results = []
    predictions_dict = {}  # Store predictions securely by ID

    for item, resp in zip(test_data, model_responses):
        is_mcq = bool(item.get("options"))
        correct = False

        # We keep this for your internal detailed_results.jsonl
        if is_mcq:
            pred_value = extract_letter(resp)
        else:
            pred_value = extract_answer_content(resp, is_mcq=False)
            if isinstance(pred_value, list):
                pred_value = str(pred_value)

        # --- CRITICAL CHANGE 1: Store the RAW response, not the extracted answer ---
        # The competition judge extracts the answer itself. It wants the full trace!
        predictions_dict[str(item["id"])] = resp
        
        if SAVE_EVAL:
            gold = item.get("answer") 
            if gold is None:
                SAVE_EVAL = False 
            if is_mcq:
                correct = extract_letter(resp) == str(gold).upper()
            elif judger:
                gold_list = gold if isinstance(gold, list) else [gold]
                try:
                    correct = judger.auto_judge(
                        pred=resp,
                        gold=gold_list,
                        options=[[]] * len(gold_list),
                    )
                except:
                    correct = False

        if not correct and SAVE_EVAL:
            print(f"\n[!] FAIL ID: {item['id']}")
            print(f" Tail of response: ...{resp[-150:]}")
            print(f" Expected: {gold}")

        results.append({
            "id": item["id"],
            "is_mcq": is_mcq,
            "response": resp,
            "gold": item.get("answer") if SAVE_EVAL else None,
            "correct": correct if SAVE_EVAL else None,
        })

    # --- CRITICAL CHANGE 2: Robust CSV generation ---
    import csv
    OUTPUT_CSV_PATH = "/kaggle/working/submission.csv"
    
    with open(OUTPUT_CSV_PATH, "w", newline='', encoding='utf-8') as f:
        # csv.QUOTE_ALL safely wraps complex math symbols and commas in quotes
        writer = csv.writer(f, quoting=csv.QUOTE_ALL)
        writer.writerow(["id", "response"])
        
        with open(DATA_PATH, "r") as f_in:
            for line in f_in:
                item = json.loads(line)
                qid = str(item["id"])
                
                # Fetch the full trace. The fallback prevents the "Null Values" error!
                full_trace = predictions_dict.get(qid, "No response generated.")
                writer.writerow([qid, full_trace])

    print(f"Submission file successfully created at: {OUTPUT_CSV_PATH}")

    # Saving your detailed results
    OUTPUT_JSONL_PATH = "/kaggle/working/detailed_results.jsonl"
    with open(OUTPUT_JSONL_PATH, "w") as f:
        for r in results:
            f.write(json.dumps(r) + "\n")

    if SAVE_EVAL:
        acc = sum(r["correct"] for r in results) / len(results) * 100
        print(f"Estimated Accuracy: {acc:.2f}%")

Generating: 100%|██████████| 118/118 [3:18:29<00:00, 100.93s/it]


Submission file successfully created at: /kaggle/working/submission.csv


In [15]:
import pandas as pd

# Load your generated submission
df = pd.read_csv("/kaggle/working/submission.csv")

# 1. Identify which column is the target
target_col = df.columns[1]

# 2. Fill empty strings or NaN values with a placeholder
# If these are MCQs, "A" is a better guess than a blank.
# If these are open-ended, "None" or "Cannot be determined" might be safer.
# For now, let's use "A" to ensure the file format is accepted.
df[target_col] = df[target_col].fillna("A")
df[target_col] = df[target_col].replace("", "A")

# 3. Verify there are no nulls left
print(f"Null values left: {df[target_col].isnull().sum()}")
print(f"Empty strings left: {(df[target_col] == '').sum()}")

# 4. Overwrite the file
df.to_csv("/kaggle/working/submission.csv", index=False)
print("File patched! Download this file from the 'Output' tab and submit it.")

Null values left: 0
Empty strings left: 0
File patched! Download this file from the 'Output' tab and submit it.
